# Replication

## 0. Configuration

In [1]:
# ============================================================
# CELL 0 — Configuration
# This is the only cell you need to edit to change models, paths, or experiment settings.
# ============================================================

import os, sys

# ── Number of questions to evaluate ─────────────────────────
NUM_QUESTIONS = 1

# ── Models ──────────────────────────────────────────────────
# All LLM calls go through this model unless overridden below.
DEFAULT_MODEL = "llama-3.3-70b-versatile"

SEARCH_QUERY_MODEL    = DEFAULT_MODEL
RANKING_MODEL         = DEFAULT_MODEL
SUMMARIZATION_MODEL   = DEFAULT_MODEL
REASONING_MODEL       = DEFAULT_MODEL

# ── Temperatures ────────────────────────────────────────────
SEARCH_QUERY_TEMPERATURE  = 0.0
RANKING_TEMPERATURE       = 0.0
SUMMARIZATION_TEMPERATURE = 0.2
REASONING_TEMPERATURE     = 1.0

# ── Retrieval settings ───────────────────────────────────────
NUM_RETRIEVAL_DATES    = 5    # max retrieval dates per question
NUM_SEARCH_QUERIES     = 3    # number of search queries to generate per retrieval date
NUM_ARTICLES_PER_QUERY = 10   # max articles to fetch per query
RELEVANCE_THRESHOLD    = 4    # 1-6 scale; keep articles rated >= this
TOP_K_ARTICLES         = 15   # keep top K after ranking
NUM_REASONING_PROMPTS  = 3    # number of scratchpad prompts to run

# ── Paths ────────────────────────────────────────────────────
REPO_ROOT    = os.path.abspath("../../llm_forecasting")   # adjust if needed
DATA_PATH    = "../../data/validation.json"               # adjust if needed
OUTPUT_DIR   = "results_02"

QUERIES_FILE     = os.path.join(OUTPUT_DIR, "queries.json")
ARTICLES_FILE    = os.path.join(OUTPUT_DIR, "articles.json")
SUMMARIES_FILE   = os.path.join(OUTPUT_DIR, "summaries.json")
PREDICTIONS_FILE = os.path.join(OUTPUT_DIR, "predictions.json")

os.makedirs(OUTPUT_DIR, exist_ok=True)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print("Config OK")
print(f"  Model:      {DEFAULT_MODEL}")
print(f"  Questions:  {NUM_QUESTIONS}")
print(f"  Data:       {DATA_PATH}")
print(f"  Output dir: {OUTPUT_DIR}")

Config OK
  Model:      llama-3.3-70b-versatile
  Questions:  1
  Data:       ../../data/validation.json
  Output dir: results_02


## 1. Imports and helpers

In [2]:
# ============================================================
# CELL 1 — Imports, LLM client, helper functions
# ============================================================

import json
import math
import time
import asyncio
import logging
from datetime import datetime, timedelta

import numpy as np
import requests
from bs4 import BeautifulSoup
from gnews import GNews
from googlenewsdecoder import gnewsdecoder

# ── Groq client (via openai-compatible API) ──────────────────
import openai
from config.keys import (
    GROQ_API_KEY_1,
    GROQ_API_KEY_2,
    GROQ_API_KEY_3,
    GROQ_API_KEY_4,
    GROQ_API_KEY_5,
)

GROQ_API_KEYS = [
    GROQ_API_KEY_1,
    GROQ_API_KEY_2,
    GROQ_API_KEY_3,
    GROQ_API_KEY_4,
    GROQ_API_KEY_5,
]

# Create one client per key
groq_clients = [
    openai.OpenAI(
        api_key=key,
        base_url="https://api.groq.com/openai/v1",
    )
    for key in GROQ_API_KEYS
]

# Track which keys are exhausted
exhausted_keys = set()

# Current active key index
current_client_idx = 0

# ── Logging ──────────────────────────────────────────────────
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# ── Core LLM call with automatic API-key rotation ────────────
def call_llm(prompt, model=None, temperature=0.0, max_tokens=4096):
    global current_client_idx, exhausted_keys

    model = model or DEFAULT_MODEL

    while True:

        # Stop if every key is exhausted
        if len(exhausted_keys) == len(groq_clients):
            raise RuntimeError("All Groq API keys are exhausted.")

        # Skip exhausted keys
        if current_client_idx in exhausted_keys:
            current_client_idx = (current_client_idx + 1) % len(groq_clients)
            continue

        client = groq_clients[current_client_idx]

        try:
            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
                max_tokens=max_tokens,
            )

            return response.choices[0].message.content

        except Exception as e:
            err = str(e).lower()

            # Detect quota / credit exhaustion
            quota_error = any(x in err for x in [
                "quota",
                "credit",
                "exceeded",
                "rate limit",
                "insufficient",
                "billing",
                "429",
            ])

            if quota_error:
                logger.warning(
                    f"API key #{current_client_idx + 1} exhausted. "
                    f"Switching to next key."
                )

                exhausted_keys.add(current_client_idx)

                current_client_idx = (
                    current_client_idx + 1
                ) % len(groq_clients)

                continue

            # Temporary failure → retry same key
            logger.warning(
                f"LLM call failed with key #{current_client_idx + 1}: {e}. "
                f"Retrying in 10s..."
            )
            time.sleep(10)

# ── JSON file helpers ─────────────────────────────────────────

def load_json(path):
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return []

def save_json(path, data):
    with open(path, "w") as f:
        json.dump(data, f, indent=2, default=str)

# ── Date helpers ─────────────────────────────────────────────

def get_retrieval_dates(date_begin, date_close, date_resolve, num_retrievals=5):
    """
    Return up to num_retrievals exponentially-spaced dates between
    date_begin and date_close, excluding any date after date_resolve.
    Mirrors the logic from the original repo's time_utils.get_retrieval_date.
    """
    date_begin_obj   = datetime.strptime(date_begin,   "%Y-%m-%d")
    date_close_obj   = datetime.strptime(date_close,   "%Y-%m-%d")
    date_resolve_obj = datetime.strptime(date_resolve, "%Y-%m-%d")

    if date_begin_obj >= date_close_obj or date_begin_obj > date_resolve_obj:
        return []

    total_days = (date_close_obj - date_begin_obj).days
    if total_days <= 0:
        return []

    dates = []
    prev_date_obj = None
    for i in range(1, num_retrievals + 1):
        days = math.exp((math.log(total_days) / num_retrievals) * i)
        d = date_begin_obj + timedelta(days=days)
        if d >= date_close_obj:
            d = date_close_obj - timedelta(days=1)
        if d >= date_resolve_obj:
            break
        if prev_date_obj and d <= prev_date_obj:
            break
        dates.append(d.strftime("%Y-%m-%d"))
        prev_date_obj = d

    return dates


def get_crowd_prediction_at_date(retrieval_date, community_predictions):
    """
    Return the crowd prediction value closest to retrieval_date.
    community_predictions is a list of [date_str, probability] pairs.
    Only considers predictions on or before retrieval_date.
    """
    ref = datetime.strptime(retrieval_date, "%Y-%m-%d")
    candidates = [
        p for p in community_predictions
        if datetime.strptime(p[0], "%Y-%m-%d") <= ref
    ]
    if not candidates:
        candidates = community_predictions
    closest = min(candidates, key=lambda p: abs((datetime.strptime(p[0], "%Y-%m-%d") - ref).days))
    return closest[1] if closest else None


def brier_score(prediction, resolution):
    return (prediction - resolution) ** 2


# ── Prediction extraction ─────────────────────────────────────

def extract_prediction(response_text):
    """
    Extract a probability (0-1) from an LLM response.
    Looks for a number wrapped in asterisks e.g. *0.75* as the prompts instruct.
    Falls back to any float in [0,1] found in the last few lines.
    Returns None if no valid probability found.
    """
    import re
    if not response_text:
        return None
    # Primary: look for *0.xx* pattern
    matches = re.findall(r"\*([01]?\.\d+)\*", response_text)
    if matches:
        return float(matches[-1])
    # Fallback: scan last 5 lines for a bare float in [0,1]
    for line in reversed(response_text.strip().split("\n")[-5:]):
        nums = re.findall(r"\b([01]?\.\d+)\b", line)
        for n in reversed(nums):
            v = float(n)
            if 0.0 <= v <= 1.0:
                return v
    return None


print("Cell 1 OK — LLM client and helpers ready")

/opt/miniconda3/envs/myenv/lib/python3.11/site-packages/newspaper/parsers.py:19: UserWarning: nltk is not installed. Some NLP features will be unavailable. Install it with: pip install 'newspaper4k[nlp]'
  from . import text as txt


Cell 1 OK — LLM client and helpers ready


## 2. Loading questions

In [3]:
# ============================================================
# CELL 2 — Load questions from validation.json
# ============================================================

with open(DATA_PATH) as f:
    all_questions = json.load(f)

questions_to_run = all_questions[:NUM_QUESTIONS]

print(f"Loaded {len(all_questions)} questions total.")
print(f"Running on {len(questions_to_run)} question(s):\n")
for i, q in enumerate(questions_to_run):
    community_preds = json.loads(q["community_predictions"])
    retrieval_dates = get_retrieval_dates(
        q["date_begin"], q["date_close"], q["date_resolve_at"], NUM_RETRIEVAL_DATES
    )
    print(f"  [{i}] {q['question']}")
    print(f"       Source:     {q['data_source']}")
    print(f"       Date range: {q['date_begin']} → {q['date_close']}")
    print(f"       Resolution: {q['resolution']} (resolved: {q['is_resolved']})")
    print(f"       Retrieval dates: {retrieval_dates}")
    print(f"       Community predictions: {len(community_preds)} data points")
    print()

Loaded 840 questions total.
Running on 1 question(s):

  [0] Will S&P 500 increase in April 2023?
       Source:     manifold
       Date range: 2023-04-03 → 2023-05-02
       Resolution: 1 (resolved: True)
       Retrieval dates: ['2023-04-04', '2023-04-06', '2023-04-10', '2023-04-17', '2023-05-01']
       Community predictions: 539 data points



## 3. Generating search queries

In [5]:
# ============================================================
# CELL 3 — Generate search queries
# Saves to queries.json. Skips questions already processed
# unless FORCE_RERUN_QUERIES = True.
# ============================================================

import re
from prompts.search_query import SEARCH_QUERY_PROMPT_0, SEARCH_QUERY_PROMPT_1

FORCE_RERUN_QUERIES = False
MAX_WORDS           = 7
NUM_KEYWORDS        = 3  # queries per prompt (6 total per retrieval date)

def fill_search_query_prompt(template, question, background, date_begin, date_end,
                              num_keywords, max_words):
    prompt_str, _ = template
    return prompt_str.format(
        question=question,
        background=background,
        date_begin=date_begin,
        date_end=date_end,
        num_keywords=num_keywords,
        max_words=max_words,
    )

def extract_queries_from_response(response_text):
    """
    Parse the semicolon-separated queries from the 'Search Queries:' section.
    Returns a list of query strings.
    """
    if not response_text:
        return []
    # Find everything after "Search Queries:"
    match = re.search(r"Search Queries:\s*(.+)", response_text, re.DOTALL | re.IGNORECASE)
    if not match:
        return []
    raw = match.group(1).strip()
    # Split on semicolons, clean up
    queries = [q.strip().strip("{}").strip() for q in raw.split(";")]
    queries = [q for q in queries if q and len(q) > 3]
    return queries

# ── Load existing data ────────────────────────────────────────
queries_data   = load_json(QUERIES_FILE)
queries_lookup = {entry["question"]: entry for entry in queries_data}

# ── Main loop ─────────────────────────────────────────────────
for q in questions_to_run:
    question   = q["question"]
    background = q["background"]
    date_close = q["date_close"]
    print(f"\nQuestion: {question[:80]}")

    retrieval_dates = get_retrieval_dates(
        q["date_begin"], q["date_close"], q["date_resolve_at"], NUM_RETRIEVAL_DATES
    )

    # Get or create entry for this question
    if question not in queries_lookup:
        queries_lookup[question] = {
            "question":               question,
            "resolution":             float(q["resolution"]),
            "date_begin":             q["date_begin"],
            "date_close":             q["date_close"],
            "date_resolve_at":        q["date_resolve_at"],
            "retrieval_dates":        retrieval_dates,
            "retrieval_date_queries": {}
        }

    entry           = queries_lookup[question]
    existing_dates  = entry["retrieval_date_queries"]

    for retrieval_date in retrieval_dates:
        if not FORCE_RERUN_QUERIES and retrieval_date in existing_dates:
            print(f"  [{retrieval_date}] Already generated — skipping.")
            continue

        print(f"  [{retrieval_date}] Generating queries...")
        date_results = {}

        for prompt_template, prompt_name in [
            (SEARCH_QUERY_PROMPT_0, "prompt_0"),
            (SEARCH_QUERY_PROMPT_1, "prompt_1"),
        ]:
            prompt = fill_search_query_prompt(
                prompt_template,
                question=question,
                background=background,
                date_begin=q["date_begin"],
                date_end=retrieval_date,
                num_keywords=NUM_KEYWORDS,
                max_words=MAX_WORDS,
            )
            response = call_llm(
                prompt,
                model=SEARCH_QUERY_MODEL,
                temperature=SEARCH_QUERY_TEMPERATURE,
            )
            queries = extract_queries_from_response(response)
            # Deduplicate across both prompts
            existing_queries = [
                qr for res in date_results.values()
                for qr in res["queries"]
            ]
            queries = [q2 for q2 in queries if q2 not in existing_queries]

            date_results[prompt_name] = {
                "llm_response": response,
                "queries":      queries,
            }
            print(f"    [{prompt_name}] {len(queries)} queries: {queries}")

        existing_dates[retrieval_date] = date_results

        # Save after every retrieval date
        save_json(QUERIES_FILE, list(queries_lookup.values()))

print(f"\nDone. Saved to {QUERIES_FILE}")


Question: Will S&P 500 increase in April 2023?
  [2023-04-04] Generating queries...


05/26/2026 01:17:38 AM - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


    [prompt_0] 3 queries: ['S&P 500 April forecast', 'US market trends 2023', 'S&P 500 investor sentiment']


05/26/2026 01:17:39 AM - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
05/26/2026 01:17:39 AM - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
05/26/2026 01:17:39 AM - Retrying request to /chat/completions in 6.000000 seconds


    [prompt_1] 2 queries: ['US economic news today', 'Market trends for April 2023']
  [2023-04-06] Generating queries...


...

    [prompt_1] 2 queries: ['US economic news today', 'Market trends this week']
  [2023-05-01] Generating queries...


05/26/2026 01:17:57 AM - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
05/26/2026 01:17:57 AM - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
05/26/2026 01:17:57 AM - Retrying request to /chat/completions in 1.000000 seconds


    [prompt_0] 3 queries: ['S&P 500 April forecast', 'US economic outlook 2023', 'Market trends April 2023']


05/26/2026 01:17:59 AM - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


    [prompt_1] 1 queries: ['Market trends for Q2 2023']

Done. Saved to results_02/queries.json


## 4. Fetching articles

In [6]:
# ============================================================
# CELL 4 — Fetch articles from GNews
# For each query, fetch top 10 results and decode URLs.
# Does NOT scrape full text yet.
# Saves to articles.json. Skips already-fetched queries
# unless FORCE_RERUN_ARTICLES = True.
# ============================================================

from gnews import GNews
from googlenewsdecoder import gnewsdecoder

FORCE_RERUN_ARTICLES = False

def fetch_gnews_articles(query, date_begin, date_end, max_results=10):
    """
    Fetch up to max_results articles from GNews for a given query
    within the date range [date_begin, date_end].
    Returns a list of raw article dicts as returned by GNews.
    """
    start = datetime.strptime(date_begin, "%Y-%m-%d")
    end   = datetime.strptime(date_end,   "%Y-%m-%d")
    gn    = GNews(
        language="en",
        country="US",
        start_date=(start.year, start.month, start.day),
        end_date=(end.year,   end.month,   end.day),
        max_results=max_results,
    )
    try:
        results = gn.get_news(query)
        return results if results else []
    except Exception as e:
        logger.warning(f"GNews fetch failed for '{query}': {e}")
        return []

def decode_google_news_url(google_url):
    """
    Decode a Google News redirect URL to the original article URL.
    Returns the decoded URL, or the original if decoding fails.
    """
    try:
        result = gnewsdecoder(google_url, interval=1)
        if result.get("status"):
            return result["decoded_url"]
    except Exception:
        pass
    return google_url

def parse_article_metadata(raw, query):
    """
    Extract clean metadata fields from a raw GNews article dict.
    Decodes the Google News URL to the real article URL.
    """
    google_url  = raw.get("url", "")
    decoded_url = decode_google_news_url(google_url)
    return {
        "title":            raw.get("title"),
        "published_date":   raw.get("published date"),
        "source":           raw.get("publisher", {}).get("title"),
        "source_url":       raw.get("publisher", {}).get("href"),
        "google_news_url":  google_url,
        "url":              decoded_url,
        "query":            query,
        "text":             None,   # filled in the next cell
    }

# ── Load existing data ────────────────────────────────────────
queries_data   = load_json(QUERIES_FILE)
articles_data  = load_json(ARTICLES_FILE)
articles_lookup = {entry["question"]: entry for entry in articles_data}

# ── Main loop ─────────────────────────────────────────────────
for q_entry in queries_data:
    question = q_entry["question"]
    print(f"\nQuestion: {question[:80]}")

    if question not in articles_lookup:
        articles_lookup[question] = {
            "question":               question,
            "resolution":             q_entry["resolution"],
            "date_begin":             q_entry["date_begin"],
            "date_close":             q_entry["date_close"],
            "date_resolve_at":        q_entry["date_resolve_at"],
            "retrieval_dates":        q_entry["retrieval_dates"],
            "retrieval_date_articles": {}
        }

    art_entry  = articles_lookup[question]
    date_art   = art_entry["retrieval_date_articles"]

    for retrieval_date, prompt_results in q_entry["retrieval_date_queries"].items():
        if retrieval_date not in date_art:
            date_art[retrieval_date] = {}

        # Collect all queries for this retrieval date across both prompts
        all_queries = []
        for prompt_name, prompt_data in prompt_results.items():
            for query in prompt_data.get("queries", []):
                if query not in all_queries:
                    all_queries.append(query)

        for query in all_queries:
            if not FORCE_RERUN_ARTICLES and query in date_art[retrieval_date]:
                print(f"  [{retrieval_date}] '{query[:55]}' — already fetched, skipping.")
                continue

            print(f"  [{retrieval_date}] Fetching: '{query[:60]}'")

            raw_articles = fetch_gnews_articles(
                query,
                date_begin=q_entry["date_begin"],
                date_end=retrieval_date,
                max_results=NUM_ARTICLES_PER_QUERY,
            )
            print(f"    GNews returned {len(raw_articles)} articles.")

            parsed = []
            for raw in raw_articles:
                meta = parse_article_metadata(raw, query)
                print(f"    → [{meta['source']}] {meta['title'][:60] if meta['title'] else 'N/A'}")
                print(f"       URL: {meta['url'][:80]}")
                parsed.append(meta)

            date_art[retrieval_date][query] = parsed

            # Save after every query
            save_json(ARTICLES_FILE, list(articles_lookup.values()))

        total = sum(len(v) for v in date_art[retrieval_date].values())
        print(f"  [{retrieval_date}] Total articles saved so far: {total}")

print(f"\nDone. Saved to {ARTICLES_FILE}")


Question: Will S&P 500 increase in April 2023?
  [2023-04-04] Fetching: 'S&P 500 April forecast'
    GNews returned 10 articles.
    → [The New York Times] In Homeless Crisis, California ‘Is Waging a War on R.V.s’ - 
       URL: https://www.nytimes.com/2026/05/24/us/california-homeless-cars-rvs.html
    → [The Washington Post] Rare glimpses inside Washington’s historic ambassador reside
       URL: https://www.washingtonpost.com/dc-md-va/interactive/2026/05/25/look-inside-some-
    → [WSJ] Opinion | L.A.’s Bonfire of Liberal Vanities - WSJ
       URL: https://www.wsj.com/opinion/l-a-s-bonfire-of-liberal-vanities-c63f9be3
    → [CBS News] President Trump won't attend son Donald Trump Jr.'s wedding,
       URL: https://www.cbsnews.com/news/trump-wont-attend-son-wedding-citing-responsibiliti
    → [Rogers-Pickard Funeral Home] Mary S. Moretz Obituary May 23, 2026 - Rogers-Pickard Funera
       URL: https://www.rogerspickard.com/obituaries/mary-moretz
    → [Thrasher Magazine] That's My S

## 5. Scraping articles' full text

In [4]:
# ============================================================
# CELL 5 — Scrape full text of articles
# Reads articles.json, scrapes full text for each unique URL,
# saves to scraped_articles.json (new file, articles.json
# is left untouched).
# Skips already-scraped URLs unless FORCE_RERUN_SCRAPING = True.
# ============================================================

import requests
from bs4 import BeautifulSoup

FORCE_RERUN_SCRAPING = False
SCRAPED_FILE         = os.path.join(OUTPUT_DIR, "scraped_articles.json")
SCRAPE_TIMEOUT       = 10   # seconds per request
SCRAPE_MIN_LENGTH    = 200  # chars; below this we treat text as failed

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

def scrape_article_text(url):
    """
    Fetch and parse the full text of an article at the given URL.
    Returns the cleaned text string, or None if:
      - the page is behind a paywall
      - the request fails
      - the extracted text is too short to be useful
    """
    if not url or url.startswith("https://news.google.com"):
        return None
    try:
        resp = requests.get(url, headers=HEADERS, timeout=SCRAPE_TIMEOUT)
        if resp.status_code != 200:
            return None
        soup = BeautifulSoup(resp.text, "html.parser")

        # Remove boilerplate tags
        for tag in soup(["script", "style", "nav", "header", "footer",
                          "aside", "form", "noscript", "iframe"]):
            tag.decompose()

        # Try to find the main article body
        article_tag = (
            soup.find("article") or
            soup.find("main") or
            soup.find("div", {"class": lambda c: c and "article" in c.lower()}) or
            soup.find("div", {"class": lambda c: c and "content" in c.lower()}) or
            soup.body
        )
        if not article_tag:
            return None

        # Extract paragraphs
        paragraphs = article_tag.find_all("p")
        text = " ".join(p.get_text(separator=" ", strip=True) for p in paragraphs)
        text = " ".join(text.split())   # collapse whitespace

        if len(text) < SCRAPE_MIN_LENGTH:
            return None
        return text

    except Exception:
        return None

# ── Load data ─────────────────────────────────────────────────
articles_data  = load_json(ARTICLES_FILE)
scraped_data   = load_json(SCRAPED_FILE)
scraped_lookup = {entry["question"]: entry for entry in scraped_data}

# ── Build a global URL cache so we never scrape the same URL twice
# across all questions / dates / queries
url_cache = {}   # url -> text or None

# Pre-populate cache from already-scraped data
for entry in scraped_data:
    for date_articles in entry.get("retrieval_date_articles", {}).values():
        for query_articles in date_articles.values():
            for art in query_articles:
                if art.get("url") and "text" in art:
                    url_cache[art["url"]] = art["text"]

# ── Main loop ─────────────────────────────────────────────────
for art_entry in articles_data:
    question = art_entry["question"]
    print(f"\n{'='*60}")
    print(f"Question: {question[:80]}")
    print('='*60)

    if question not in scraped_lookup:
        scraped_lookup[question] = {
            "question":               question,
            "resolution":             art_entry["resolution"],
            "date_begin":             art_entry["date_begin"],
            "date_close":             art_entry["date_close"],
            "date_resolve_at":        art_entry["date_resolve_at"],
            "retrieval_dates":        art_entry["retrieval_dates"],
            "retrieval_date_articles": {}
        }

    scraped_entry = scraped_lookup[question]
    date_art_out  = scraped_entry["retrieval_date_articles"]

    for retrieval_date, query_articles in art_entry["retrieval_date_articles"].items():
        if retrieval_date not in date_art_out:
            date_art_out[retrieval_date] = {}

        print(f"\n  [{retrieval_date}]")

        for query, articles in query_articles.items():
            if not FORCE_RERUN_SCRAPING and query in date_art_out[retrieval_date]:
                already = len(date_art_out[retrieval_date][query])
                print(f"    '{query[:55]}' — already scraped ({already} articles), skipping.")
                continue

            print(f"    Scraping: '{query[:60]}'  ({len(articles)} articles)")
            scraped_articles = []

            for i, art in enumerate(articles):
                url = art.get("url")
                title = art.get("title", "")[:55] or "N/A"

                if not FORCE_RERUN_SCRAPING and url in url_cache:
                    text = url_cache[url]
                    status = "cached"
                else:
                    text = scrape_article_text(url)
                    url_cache[url] = text
                    status = "scraped" if text else "failed"
                    time.sleep(0.5)   # be polite

                indicator = "✓" if text else "✗"
                print(f"      [{i+1}/{len(articles)}] {indicator} [{status}] {title}")

                scraped_articles.append({**art, "text": text})

            date_art_out[retrieval_date][query] = scraped_articles

            # Save after every query
            save_json(SCRAPED_FILE, list(scraped_lookup.values()))

# ── Final summary ─────────────────────────────────────────────
print(f"\n{'='*60}")
print("SUMMARY")
print('='*60)
total_articles  = 0
missing_text    = 0
for entry in scraped_lookup.values():
    for date_articles in entry["retrieval_date_articles"].values():
        for query_articles in date_articles.values():
            for art in query_articles:
                total_articles += 1
                if not art.get("text"):
                    missing_text += 1

print(f"  Total articles : {total_articles}")
print(f"  With full text : {total_articles - missing_text}")
print(f"  Missing text   : {missing_text}  ({100*missing_text/total_articles:.1f}% of total)" if total_articles else "  No articles.")
print(f"\nDone. Saved to {SCRAPED_FILE}")


Question: Will S&P 500 increase in April 2023?

  [2023-04-04]
    'S&P 500 April forecast' — already scraped (10 articles), skipping.
    'US market trends 2023' — already scraped (9 articles), skipping.
    'S&P 500 investor sentiment' — already scraped (10 articles), skipping.
    'US economic news today' — already scraped (10 articles), skipping.
    'Market trends for April 2023' — already scraped (10 articles), skipping.

  [2023-04-06]
    'S&P 500 April forecast' — already scraped (10 articles), skipping.
    'US market trends 2023' — already scraped (10 articles), skipping.
    'S&P 500 analyst predictions' — already scraped (10 articles), skipping.
    'US economic outlook' — already scraped (10 articles), skipping.
    'Market trends 2023' — already scraped (10 articles), skipping.

  [2023-04-10]
    'S&P 500 April forecast' — already scraped (10 articles), skipping.
    'US market trends 2023' — already scraped (10 articles), skipping.
    'S&P 500 investor sentiment' — a

## 6. Ranking relevance

In [5]:
# ============================================================
# CELL 6 — Relevance ranking
# Reads scraped_articles.json, asks the LLM to rate each
# article's relevance (1-6), keeps those >= RELEVANCE_THRESHOLD,
# saves top K per retrieval date to ranked_articles.json.
# Skips already-ranked queries unless FORCE_RERUN_RANKING = True.
# ============================================================

import re
from prompts.relevance import RELEVANCE_PROMPT_0

FORCE_RERUN_RANKING = True
RANKED_FILE         = os.path.join(OUTPUT_DIR, "ranked_articles.json")

# ── We need full question metadata for prompts ────────────────
q_meta = {q["question"]: q for q in questions_to_run}

def fill_relevance_prompt(article_text, question, background, resolution_criteria):
    prompt_str, _ = RELEVANCE_PROMPT_0
    # Use title + text as the article content, fall back to description
    return prompt_str.format(
        question=question,
        background=background,
        resolution_criteria=resolution_criteria,
        article=article_text,
    )

def extract_relevance_rating(response_text):
    """
    Extract the integer rating (1-6) from the LLM relevance response.
    Looks for 'Rating: N' pattern. Returns None if not found.
    """
    if not response_text:
        return None
    match = re.search(r"Rating:\s*([1-6])", response_text)
    if match:
        return int(match.group(1))
    # Fallback: look for a standalone digit 1-6 near the end
    lines = response_text.strip().split("\n")
    for line in reversed(lines[-5:]):
        m = re.search(r"\b([1-6])\b", line)
        if m:
            return int(m.group(1))
    return None

def build_article_text_for_ranking(art):
    """
    Build the text to send to the LLM for ranking.
    Uses full text if available, otherwise falls back to title + description.
    """
    parts = []
    if art.get("title"):
        parts.append(f"Title: {art['title']}")
    if art.get("text"):
        # Truncate to ~2000 chars to save tokens during ranking
        parts.append(art["text"][:2000])
    elif art.get("description"):
        parts.append(f"Description: {art['description']}")
    return "\n".join(parts) if parts else None

# ── Load data ─────────────────────────────────────────────────
scraped_data   = load_json(SCRAPED_FILE)
ranked_data    = load_json(RANKED_FILE)
ranked_lookup  = {entry["question"]: entry for entry in ranked_data}

# ── Main loop ─────────────────────────────────────────────────
for scraped_entry in scraped_data:
    question = scraped_entry["question"]
    q_info   = q_meta.get(question, {})
    print(f"\n{'='*60}")
    print(f"Question: {question[:80]}")
    print('='*60)

    if question not in ranked_lookup:
        ranked_lookup[question] = {
            "question":                  question,
            "resolution":                scraped_entry["resolution"],
            "date_begin":                scraped_entry["date_begin"],
            "date_close":                scraped_entry["date_close"],
            "date_resolve_at":           scraped_entry["date_resolve_at"],
            "retrieval_dates":           scraped_entry["retrieval_dates"],
            "retrieval_date_rankings":   {}
        }

    ranked_entry   = ranked_lookup[question]
    date_rankings  = ranked_entry["retrieval_date_rankings"]

    for retrieval_date, query_articles in scraped_entry["retrieval_date_articles"].items():
        if not FORCE_RERUN_RANKING and retrieval_date in date_rankings:
            n = date_rankings[retrieval_date]["num_articles_ranked"]
            print(f"\n  [{retrieval_date}] Already ranked ({n} articles kept) — skipping.")
            continue

        print(f"\n  [{retrieval_date}] Ranking articles...")

        # Pool all articles across queries, deduplicate by URL
        seen_urls    = set()
        pooled       = []
        for query, articles in query_articles.items():
            for art in articles:
                url = art.get("url")
                if url and url not in seen_urls:
                    seen_urls.add(url)
                    pooled.append(art)

        print(f"    Pooled {len(pooled)} unique articles across all queries.")

        if not pooled:
            print("    No articles to rank — saving empty entry.")
            date_rankings[retrieval_date] = {
                "num_articles_pooled":  0,
                "num_articles_rated":   0,
                "num_articles_ranked":  0,
                "all_rated_articles":   [],
                "ranked_articles":      [],
            }
            save_json(RANKED_FILE, list(ranked_lookup.values()))
            continue

        # Rate each article
        all_rated = []
        for i, art in enumerate(pooled):
            title     = (art.get("title") or "")[:55]
            art_text  = build_article_text_for_ranking(art)

            if not art_text:
                print(f"    [{i+1}/{len(pooled)}] ✗ No content — skipping: {title}")
                all_rated.append({
                    **art,
                    "relevance_rating":   None,
                    "relevance_response": None,
                })
                continue

            prompt   = fill_relevance_prompt(
                article_text=art_text,
                question=question,
                background=q_info.get("background", ""),
                resolution_criteria=q_info.get("resolution_criteria", ""),
            )
            response = call_llm(
                prompt,
                model=RANKING_MODEL,
                temperature=RANKING_TEMPERATURE,
            )
            rating = extract_relevance_rating(response)
            indicator = f"★{rating}" if rating is not None else "??"
            print(f"    [{i+1}/{len(pooled)}] [{indicator}] {title}")

            all_rated.append({
                **art,
                "relevance_rating":   rating,
                "relevance_response": response,
            })

        # Filter by threshold and keep top K
        ranked = [
            a for a in all_rated
            if a["relevance_rating"] is not None
            and a["relevance_rating"] >= RELEVANCE_THRESHOLD
        ]
        # Sort by rating descending, then take top K
        ranked = sorted(ranked, key=lambda a: a["relevance_rating"], reverse=True)
        ranked = ranked[:TOP_K_ARTICLES]

        print(f"    {len(all_rated)} rated → {len(ranked)} kept "
              f"(threshold={RELEVANCE_THRESHOLD}, top_k={TOP_K_ARTICLES})")

        date_rankings[retrieval_date] = {
            "num_articles_pooled":  len(pooled),
            "num_articles_rated":   len(all_rated),
            "num_articles_ranked":  len(ranked),
            "all_rated_articles":   all_rated,    # full list with scores + LLM responses
            "ranked_articles":      ranked,        # filtered + sorted top K
        }

        save_json(RANKED_FILE, list(ranked_lookup.values()))
        print(f"    Saved to {RANKED_FILE}")

# ── Summary ───────────────────────────────────────────────────
print(f"\n{'='*60}")
print("SUMMARY")
print('='*60)
for entry in ranked_lookup.values():
    print(f"\n  {entry['question'][:70]}")
    for date, dr in entry["retrieval_date_rankings"].items():
        print(f"    [{date}] pooled={dr['num_articles_pooled']}  "
              f"rated={dr['num_articles_rated']}  "
              f"kept={dr['num_articles_ranked']}")

print(f"\nDone. Saved to {RANKED_FILE}")


Question: Will S&P 500 increase in April 2023?

  [2023-04-04] Ranking articles...
    Pooled 37 unique articles across all queries.


05/26/2026 02:10:50 AM - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


    [1/37] [★1] In Homeless Crisis, California ‘Is Waging a War on R.V.


05/26/2026 02:10:51 AM - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
05/26/2026 02:10:51 AM - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
05/26/2026 02:10:51 AM - Retrying request to /chat/completions in 7.000000 seconds


    [2/37] [★1] Rare glimpses inside Washington’s historic ambassador r


...

    [38/40] [★2] Amazon Q1 2023: Top 25 Beauty and Personal Care Product


05/26/2026 02:20:14 AM - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
05/26/2026 02:20:14 AM - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
05/26/2026 02:20:14 AM - Retrying request to /chat/completions in 4.000000 seconds


    [39/40] [★1] Southeast Asia Outlook 2026 | Market & Investment Trend


05/26/2026 02:20:19 AM - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


    [40/40] [★1] What’s really going on with Twitter? - DataReportal – G
    40 rated → 0 kept (threshold=4, top_k=15)
    Saved to results_02/ranked_articles.json

SUMMARY

  Will S&P 500 increase in April 2023?
    [2023-04-04] pooled=37  rated=37  kept=2
    [2023-04-06] pooled=34  rated=34  kept=3
    [2023-04-10] pooled=34  rated=34  kept=1
    [2023-04-17] pooled=40  rated=40  kept=2
    [2023-05-01] pooled=40  rated=40  kept=0

Done. Saved to results_02/ranked_articles.json


## 7. Summarising

In [6]:
# ============================================================
# CELL 7 — Summarise ranked articles
# Reads ranked_articles.json, summarises each unique article
# (only those that passed the relevance threshold).
# Saves to summaries.json (new file).
# Skips already-summarised articles unless FORCE_RERUN = True.
# ============================================================

from prompts.summarization import SUMMARIZATION_PROMPT_0

FORCE_RERUN_SUMMARIES = False
SUMMARIES_FILE        = os.path.join(OUTPUT_DIR, "summaries.json")

def fill_summarization_prompt(article_text, question, background):
    prompt_str, _ = SUMMARIZATION_PROMPT_0
    return prompt_str.format(
        question=question,
        background=background,
        article=article_text,
    )

# ── Load data ─────────────────────────────────────────────────
ranked_data      = load_json(RANKED_FILE)
summaries_data   = load_json(SUMMARIES_FILE)
summaries_lookup = {entry["question"]: entry for entry in summaries_data}

# ── Build a global URL -> summary cache so each article
#    is only summarised once across all dates/queries
summary_cache = {}   # url -> summary text
for entry in summaries_data:
    for date_data in entry.get("retrieval_date_summaries", {}).values():
        for art in date_data.get("summarised_articles", []):
            if art.get("url") and art.get("summary"):
                summary_cache[art["url"]] = art["summary"]

q_meta = {q["question"]: q for q in questions_to_run}

# ── Main loop ─────────────────────────────────────────────────
for ranked_entry in ranked_data:
    question = ranked_entry["question"]
    q_info   = q_meta.get(question, {})
    print(f"\n{'='*60}")
    print(f"Question: {question[:80]}")
    print('='*60)

    if question not in summaries_lookup:
        summaries_lookup[question] = {
            "question":                  question,
            "resolution":                ranked_entry["resolution"],
            "date_begin":                ranked_entry["date_begin"],
            "date_close":                ranked_entry["date_close"],
            "date_resolve_at":           ranked_entry["date_resolve_at"],
            "retrieval_dates":           ranked_entry["retrieval_dates"],
            "retrieval_date_summaries":  {}
        }

    sum_entry  = summaries_lookup[question]
    date_sums  = sum_entry["retrieval_date_summaries"]

    for retrieval_date, date_data in ranked_entry["retrieval_date_rankings"].items():
        if not FORCE_RERUN_SUMMARIES and retrieval_date in date_sums:
            n = date_sums[retrieval_date]["num_summarised"]
            print(f"\n  [{retrieval_date}] Already summarised ({n} articles) — skipping.")
            continue

        ranked_articles = date_data.get("ranked_articles", [])
        print(f"\n  [{retrieval_date}] {len(ranked_articles)} articles to summarise.")

        if not ranked_articles:
            print("    No ranked articles — saving empty entry.")
            date_sums[retrieval_date] = {
                "num_ranked":        0,
                "num_summarised":    0,
                "summarised_articles": [],
                "combined_summary":  "",
            }
            save_json(SUMMARIES_FILE, list(summaries_lookup.values()))
            continue

        summarised = []
        for i, art in enumerate(ranked_articles):
            url   = art.get("url", "")
            title = (art.get("title") or "")[:60]
            text  = art.get("text")

            # Skip articles with no text
            if not text:
                print(f"    [{i+1}/{len(ranked_articles)}] ✗ No text — skipping: {title}")
                summarised.append({**art, "summary": None, "summary_llm_response": None})
                continue

            # Use cached summary if available
            if not FORCE_RERUN_SUMMARIES and url in summary_cache:
                print(f"    [{i+1}/{len(ranked_articles)}] ↩ Cached: {title}")
                summarised.append({**art, "summary": summary_cache[url], "summary_llm_response": None})
                continue

            # Call LLM
            prompt   = fill_summarization_prompt(
                article_text=text[:4000],  # truncate to save tokens
                question=question,
                background=q_info.get("background", ""),
            )
            response = call_llm(
                prompt,
                model=SUMMARIZATION_MODEL,
                temperature=SUMMARIZATION_TEMPERATURE,
                max_tokens=512,
            )
            summary = response.strip() if response else None
            summary_cache[url] = summary

            status = "✓" if summary else "✗"
            print(f"    [{i+1}/{len(ranked_articles)}] {status} Summarised: {title}")

            summarised.append({
                **art,
                "summary":              summary,
                "summary_llm_response": response,
            })

        # Build combined summary string to feed into reasoning prompts
        # Format: --- Article N ---\nTitle: ...\nSummary: ...\n
        combined_parts = []
        for i, art in enumerate(summarised):
            if art.get("summary"):
                combined_parts.append(
                    f"--- Article {i+1} ---\n"
                    f"Title: {art.get('title', 'N/A')}\n"
                    f"Source: {art.get('source', 'N/A')}\n"
                    f"Date: {art.get('published_date', 'N/A')}\n"
                    f"Summary: {art['summary']}"
                )
        combined_summary = "\n\n".join(combined_parts)

        num_summarised = len([a for a in summarised if a.get("summary")])
        print(f"    {num_summarised}/{len(ranked_articles)} articles successfully summarised.")
        print(f"    Combined summary: {len(combined_summary)} chars.")

        date_sums[retrieval_date] = {
            "num_ranked":          len(ranked_articles),
            "num_summarised":      num_summarised,
            "summarised_articles": summarised,
            "combined_summary":    combined_summary,
        }

        save_json(SUMMARIES_FILE, list(summaries_lookup.values()))

# ── Summary ───────────────────────────────────────────────────
print(f"\n{'='*60}")
print("SUMMARY")
print('='*60)
for entry in summaries_lookup.values():
    print(f"\n  {entry['question'][:70]}")
    for date, ds in entry["retrieval_date_summaries"].items():
        print(f"    [{date}] ranked={ds['num_ranked']}  "
              f"summarised={ds['num_summarised']}  "
              f"combined={len(ds['combined_summary'])} chars")

print(f"\nDone. Saved to {SUMMARIES_FILE}")


Question: Will S&P 500 increase in April 2023?

  [2023-04-04] 2 articles to summarise.


05/26/2026 02:27:36 AM - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


    [1/2] ✓ Summarised: Asian shares mixed after Wall St dips on weak economic data 
    [2/2] ✗ No text — skipping: U.S. dollar slumps after weak data; markets betting Fed near
    1/2 articles successfully summarised.
    Combined summary: 2201 chars.

  [2023-04-06] 3 articles to summarise.


05/26/2026 02:27:37 AM - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


    [1/3] ✓ Summarised: Recession Risks: The Global Impact of a US Hard Landing Scen
    [2/3] ✗ No text — skipping: U.S. dollar slumps after weak data; markets betting Fed near
    [3/3] ✗ No text — skipping: There are 3 key signs that suggest a recession is around the
    1/3 articles successfully summarised.
    Combined summary: 1932 chars.

  [2023-04-10] 1 articles to summarise.
    [1/1] ↩ Cached: Recession Risks: The Global Impact of a US Hard Landing Scen
    1/1 articles successfully summarised.
    Combined summary: 1932 chars.

  [2023-04-17] 2 articles to summarise.
    [1/2] ✗ No text — skipping: US economy still churning out jobs at brisk clip; wage press


05/26/2026 02:27:38 AM - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


    [2/2] ✓ Summarised: Survey: Fed To Hike Rates Once More Before 2024 Cuts - Bankr
    1/2 articles successfully summarised.
    Combined summary: 2096 chars.

  [2023-05-01] 0 articles to summarise.
    No ranked articles — saving empty entry.

SUMMARY

  Will S&P 500 increase in April 2023?
    [2023-04-04] ranked=2  summarised=1  combined=2201 chars
    [2023-04-06] ranked=3  summarised=1  combined=1932 chars
    [2023-04-10] ranked=1  summarised=1  combined=1932 chars
    [2023-04-17] ranked=2  summarised=1  combined=2096 chars
    [2023-05-01] ranked=0  summarised=0  combined=0 chars

Done. Saved to results_02/summaries.json


## 8. Getting probabilities

In [7]:
# ============================================================
# CELL 8 — Generate probabilistic forecasts
# Reads summaries.json, feeds combined article summaries into
# the reasoning prompt, extracts probabilities, and saves
# full reasoning traces + predictions to predictions.json.
#
# Runs once per retrieval date.
# Skips already-generated forecasts unless
# FORCE_RERUN_PREDICTIONS = True.
# ============================================================

import re
from prompts.base_reasoning import BINARY_SCRATCH_PAD_PROMPT_0

FORCE_RERUN_PREDICTIONS = False
PREDICTIONS_FILE        = os.path.join(OUTPUT_DIR, "predictions.json")

def fill_reasoning_prompt(
    question,
    background,
    resolution_criteria,
    date_begin,
    date_end,
    retrieved_info,
):
    """
    Fill the binary forecasting scratchpad prompt.
    """
    prompt_str, _ = BINARY_SCRATCH_PAD_PROMPT_0

    return prompt_str.format(
        question=question,
        background=background,
        resolution_criteria=resolution_criteria,
        date_begin=date_begin,
        date_end=date_end,
        retrieved_info=retrieved_info,
    )

# ── Load data ─────────────────────────────────────────────────
summaries_data     = load_json(SUMMARIES_FILE)
predictions_data   = load_json(PREDICTIONS_FILE)
predictions_lookup = {entry["question"]: entry for entry in predictions_data}

# ── Full metadata lookup ─────────────────────────────────────
q_meta = {q["question"]: q for q in questions_to_run}

# ── Main loop ─────────────────────────────────────────────────
for sum_entry in summaries_data:

    question = sum_entry["question"]
    q_info   = q_meta.get(question, {})

    print(f"\n{'='*60}")
    print(f"Question: {question[:80]}")
    print('='*60)

    # Create prediction entry if missing
    if question not in predictions_lookup:
        predictions_lookup[question] = {
            "question":                   question,
            "resolution":                 sum_entry["resolution"],
            "date_begin":                 sum_entry["date_begin"],
            "date_close":                 sum_entry["date_close"],
            "date_resolve_at":            sum_entry["date_resolve_at"],
            "retrieval_dates":            sum_entry["retrieval_dates"],
            "retrieval_date_predictions": {}
        }

    pred_entry = predictions_lookup[question]
    date_preds = pred_entry["retrieval_date_predictions"]

    # ── Iterate over retrieval dates ─────────────────────────
    for retrieval_date, date_data in sum_entry["retrieval_date_summaries"].items():

        if not FORCE_RERUN_PREDICTIONS and retrieval_date in date_preds:
            existing_prob = date_preds[retrieval_date].get("prediction")
            print(
                f"\n  [{retrieval_date}] "
                f"Already predicted ({existing_prob}) — skipping."
            )
            continue

        print(f"\n  [{retrieval_date}] Generating forecast...")

        combined_summary = date_data.get("combined_summary", "").strip()

        if not combined_summary:
            print("    ✗ No summaries available — saving empty prediction.")

            date_preds[retrieval_date] = {
                "num_articles_used": 0,
                "retrieved_info_chars": 0,
                "llm_response": None,
                "prediction": None,
            }

            save_json(PREDICTIONS_FILE, list(predictions_lookup.values()))
            continue

        # Build reasoning prompt
        prompt = fill_reasoning_prompt(
            question=question,
            background=q_info.get("background", ""),
            resolution_criteria=q_info.get("resolution_criteria", ""),
            date_begin=retrieval_date,
            date_end=q_info.get("date_close", ""),
            retrieved_info=combined_summary,
        )

        # Call LLM
        response = call_llm(
            prompt,
            model=REASONING_MODEL,
            temperature=REASONING_TEMPERATURE,
            max_tokens=2048,
        )

        # Extract probability
        prediction = extract_prediction(response)

        indicator = (
            f"{prediction:.3f}"
            if prediction is not None
            else "FAILED"
        )

        print(f"    Prediction: {indicator}")

        # Save full reasoning trace + extracted probability
        date_preds[retrieval_date] = {
            "num_articles_used": date_data.get("num_summarised", 0),
            "retrieved_info_chars": len(combined_summary),
            "llm_response": response,
            "prediction": prediction,
        }

        save_json(PREDICTIONS_FILE, list(predictions_lookup.values()))

# ── Summary ───────────────────────────────────────────────────
print(f"\n{'='*60}")
print("SUMMARY")
print('='*60)

for entry in predictions_lookup.values():

    print(f"\n  {entry['question'][:70]}")

    for retrieval_date, pred in entry["retrieval_date_predictions"].items():

        prediction = pred.get("prediction")

        pred_str = (
            f"{prediction:.3f}"
            if prediction is not None
            else "None"
        )

        print(
            f"    [{retrieval_date}] "
            f"prediction={pred_str}  "
            f"articles={pred.get('num_articles_used', 0)}"
        )

print(f"\nDone. Saved to {PREDICTIONS_FILE}")


Question: Will S&P 500 increase in April 2023?

  [2023-04-04] Generating forecast...


05/26/2026 02:29:26 AM - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


    Prediction: 0.550

  [2023-04-06] Generating forecast...


05/26/2026 02:29:28 AM - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


    Prediction: 0.400

  [2023-04-10] Generating forecast...


05/26/2026 02:29:30 AM - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


    Prediction: 0.450

  [2023-04-17] Generating forecast...


05/26/2026 02:29:33 AM - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


    Prediction: 0.550

  [2023-05-01] Generating forecast...
    ✗ No summaries available — saving empty prediction.

SUMMARY

  Will S&P 500 increase in April 2023?
    [2023-04-04] prediction=0.550  articles=1
    [2023-04-06] prediction=0.400  articles=1
    [2023-04-10] prediction=0.450  articles=1
    [2023-04-17] prediction=0.550  articles=1
    [2023-05-01] prediction=None  articles=0

Done. Saved to results_02/predictions.json


## 9. Analysing results

In [8]:
# ============================================================
# CELL 9 — Compare LLM vs Crowd Brier scores
# Reads predictions.json and compares:
#   - LLM prediction Brier score
#   - Crowd prediction Brier score
# at each retrieval date.
# ============================================================

predictions_data = load_json(PREDICTIONS_FILE)

print(f"\n{'='*120}")
print("LLM vs CROWD — BRIER SCORE COMPARISON")
print('='*120)

for entry in predictions_data:

    question   = entry["question"]
    resolution = float(entry["resolution"])

    # Load original crowd predictions
    q_original = next(
        q for q in all_questions
        if q["question"] == question
    )

    community_predictions = json.loads(
        q_original["community_predictions"]
    )

    print(f"\nQuestion: {question}")
    print("-" * 120)

    header = (
        f"{'Retrieval Date':<15}"
        f"{'LLM Pred':>12}"
        f"{'Crowd Pred':>15}"
        f"{'LLM Brier':>15}"
        f"{'Crowd Brier':>15}"
    )

    print(header)
    print("-" * len(header))

    for retrieval_date, pred_data in entry["retrieval_date_predictions"].items():

        llm_pred = pred_data.get("prediction")

        crowd_pred = get_crowd_prediction_at_date(
            retrieval_date,
            community_predictions
        )

        llm_brier = (
            brier_score(llm_pred, resolution)
            if llm_pred is not None
            else None
        )

        crowd_brier = (
            brier_score(crowd_pred, resolution)
            if crowd_pred is not None
            else None
        )

        print(
            f"{retrieval_date:<15}"
            f"{str(round(llm_pred, 4)) if llm_pred is not None else 'None':>12}"
            f"{str(round(crowd_pred, 4)) if crowd_pred is not None else 'None':>15}"
            f"{str(round(llm_brier, 4)) if llm_brier is not None else 'None':>15}"
            f"{str(round(crowd_brier, 4)) if crowd_brier is not None else 'None':>15}"
        )

print(f"\n{'='*120}")
print("Done.")
print('='*120)


LLM vs CROWD — BRIER SCORE COMPARISON

Question: Will S&P 500 increase in April 2023?
------------------------------------------------------------------------------------------------------------------------
Retrieval Date     LLM Pred     Crowd Pred      LLM Brier    Crowd Brier
------------------------------------------------------------------------
2023-04-04             0.55         0.5651         0.2025         0.1892
2023-04-06              0.4         0.5621           0.36         0.1918
2023-04-10             0.45         0.5687         0.3025          0.186
2023-04-17             0.55         0.6641         0.2025         0.1128
2023-05-01             None         0.9971           None            0.0

Done.
